# Lab 9 — Exportação da camada Gold

## Objetivo

Este laboratório demonstra como disponibilizar os dados agregados da camada Gold para consumo externo.

Serão testados dois caminhos:

1. exportação direta das tabelas Gold para CSV;
2. carga das tabelas em um banco relacional SQLite.

A exportação em CSV será utilizada posteriormente no dashboard por ser simples, portátil e compatível com a rota sem privilégios administrativos. O SQLite representa uma alternativa de serving por banco relacional, semelhante conceitualmente a uma exportação do Hive para MySQL por Sqoop.

In [1]:
from pathlib import Path
import sqlite3
import duckdb
import pandas as pd

pasta_projeto = Path.cwd().resolve()

if not (pasta_projeto / "dados" / "gold").exists():
    for pasta_pai in pasta_projeto.parents:
        if (pasta_pai / "dados" / "gold").exists():
            pasta_projeto = pasta_pai
            break

arquivo_gold_segmento = (
    pasta_projeto / "dados" / "gold" / "fraud_risk.parquet"
)

arquivo_gold_diario = (
    pasta_projeto / "dados" / "gold" / "daily_metrics.parquet"
)

pasta_serving = pasta_projeto / "dados" / "serving"
pasta_serving.mkdir(parents=True, exist_ok=True)

arquivo_csv_segmento = (
    pasta_serving / "fraud_risk_export.csv"
)

arquivo_csv_diario = (
    pasta_serving / "daily_metrics_export.csv"
)

arquivo_sqlite = pasta_serving / "bi_db.sqlite"

arquivo_duckdb = (
    pasta_projeto
    / "dia3_insights_bi"
    / "lab09_export"
    / "export.duckdb"
)

assert arquivo_gold_segmento.exists(), (
    "Gold por segmento não encontrada."
)

assert arquivo_gold_diario.exists(), (
    "Gold diária não encontrada."
)

print("Gold por segmento:", arquivo_gold_segmento)
print("Gold diária:", arquivo_gold_diario)
print("Pasta de serving:", pasta_serving)

Gold por segmento: C:\BigData\bigdata-curso-gabriel\dados\gold\fraud_risk.parquet
Gold diária: C:\BigData\bigdata-curso-gabriel\dados\gold\daily_metrics.parquet
Pasta de serving: C:\BigData\bigdata-curso-gabriel\dados\serving


In [2]:
conexao_duckdb = duckdb.connect(str(arquivo_duckdb))

caminho_gold_segmento = (
    arquivo_gold_segmento.as_posix().replace("'", "''")
)

caminho_gold_diario = (
    arquivo_gold_diario.as_posix().replace("'", "''")
)

conexao_duckdb.execute(f"""
    CREATE OR REPLACE TABLE gold_fraud_risk AS
    SELECT *
    FROM read_parquet('{caminho_gold_segmento}')
""")

conexao_duckdb.execute(f"""
    CREATE OR REPLACE TABLE gold_daily_metrics AS
    SELECT *
    FROM read_parquet('{caminho_gold_diario}')
""")

tabelas_origem = conexao_duckdb.execute("""
    SELECT
        'gold_fraud_risk' AS tabela,
        COUNT(*) AS linhas
    FROM gold_fraud_risk

    UNION ALL

    SELECT
        'gold_daily_metrics' AS tabela,
        COUNT(*) AS linhas
    FROM gold_daily_metrics
""").df()

tabelas_origem

,tabela,linhas
0,gold_fraud_risk,3
1,gold_daily_metrics,672


## Caminho 1 - Exportação para CSV

In [3]:
for arquivo in [arquivo_csv_segmento, arquivo_csv_diario]:
    if arquivo.exists():
        arquivo.unlink()

destino_csv_segmento = (
    arquivo_csv_segmento.as_posix().replace("'", "''")
)

destino_csv_diario = (
    arquivo_csv_diario.as_posix().replace("'", "''")
)

conexao_duckdb.execute(f"""
    COPY gold_fraud_risk
    TO '{destino_csv_segmento}'
    (
        FORMAT CSV,
        HEADER,
        DELIMITER ','
    )
""")

conexao_duckdb.execute(f"""
    COPY gold_daily_metrics
    TO '{destino_csv_diario}'
    (
        FORMAT CSV,
        HEADER,
        DELIMITER ','
    )
""")

print("Arquivos CSV exportados:")
print("-", arquivo_csv_segmento)
print("-", arquivo_csv_diario)

Arquivos CSV exportados:
- C:\BigData\bigdata-curso-gabriel\dados\serving\fraud_risk_export.csv
- C:\BigData\bigdata-curso-gabriel\dados\serving\daily_metrics_export.csv


In [4]:
csv_segmento = pd.read_csv(arquivo_csv_segmento)
csv_diario = pd.read_csv(arquivo_csv_diario)

validacao_csv = pd.DataFrame({
    "Tabela": [
        "Gold por segmento",
        "Gold diária"
    ],
    "Linhas na origem": [
        3,
        672
    ],
    "Linhas no CSV": [
        len(csv_segmento),
        len(csv_diario)
    ]
})

validacao_csv["Validação"] = (
    validacao_csv["Linhas na origem"]
    == validacao_csv["Linhas no CSV"]
).map({
    True: "OK",
    False: "DIVERGENTE"
})

validacao_csv

,Tabela,Linhas na origem,Linhas no CSV,Validação
0,Gold por segmento,3,3,OK
1,Gold diária,672,672,OK


In [5]:
csv_segmento.sort_values(
    "taxa_fraude_pct",
    ascending=False
)

,segment,total_transacoes,total_clientes,valor_total,ticket_medio,qtd_fraudes,taxa_fraude_pct,valor_em_risco
2,High-Risk,9155,876,1664640.24,181.83,705.0,7.70,126451.07
0,Standard,29689,2665,5479399.85,184.56,655.0,2.21,106828.42
1,Premium,61156,5563,11235041.58,183.71,473.0,0.77,81308.44


## Caminho 2 - Exportação para SQLite

In [6]:
gold_segmento_df = conexao_duckdb.execute("""
    SELECT *
    FROM gold_fraud_risk
""").df()

gold_diario_df = conexao_duckdb.execute("""
    SELECT *
    FROM gold_daily_metrics
""").df()

with sqlite3.connect(arquivo_sqlite) as conexao_sqlite:
    gold_segmento_df.to_sql(
        "fraud_risk_bi",
        conexao_sqlite,
        if_exists="replace",
        index=False
    )

    gold_diario_df.to_sql(
        "daily_metrics_bi",
        conexao_sqlite,
        if_exists="replace",
        index=False
    )

print("Tabelas exportadas para:", arquivo_sqlite)

Tabelas exportadas para: C:\BigData\bigdata-curso-gabriel\dados\serving\bi_db.sqlite


In [7]:
with sqlite3.connect(arquivo_sqlite) as conexao_sqlite:
    tabelas_sqlite = pd.read_sql_query(
        """
        SELECT
            name AS tabela
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        conexao_sqlite
    )

tabelas_sqlite

,tabela
0,daily_metrics_bi
1,fraud_risk_bi


In [8]:
with sqlite3.connect(arquivo_sqlite) as conexao_sqlite:
    linhas_segmento_sqlite = conexao_sqlite.execute(
        "SELECT COUNT(*) FROM fraud_risk_bi"
    ).fetchone()[0]

    linhas_diario_sqlite = conexao_sqlite.execute(
        "SELECT COUNT(*) FROM daily_metrics_bi"
    ).fetchone()[0]

validacao_sqlite = pd.DataFrame({
    "Tabela": [
        "fraud_risk_bi",
        "daily_metrics_bi"
    ],
    "Linhas esperadas": [
        len(gold_segmento_df),
        len(gold_diario_df)
    ],
    "Linhas no SQLite": [
        linhas_segmento_sqlite,
        linhas_diario_sqlite
    ]
})

validacao_sqlite["Validação"] = (
    validacao_sqlite["Linhas esperadas"]
    == validacao_sqlite["Linhas no SQLite"]
).map({
    True: "OK",
    False: "DIVERGENTE"
})

validacao_sqlite

,Tabela,Linhas esperadas,Linhas no SQLite,Validação
0,fraud_risk_bi,3,3,OK
1,daily_metrics_bi,672,672,OK


In [9]:
with sqlite3.connect(arquivo_sqlite) as conexao_sqlite:
    totais_sqlite = pd.read_sql_query(
        """
        SELECT
            SUM(total_transacoes) AS transacoes,
            SUM(qtd_fraudes) AS fraudes,
            ROUND(SUM(valor_em_risco), 2) AS valor_em_risco
        FROM fraud_risk_bi
        """,
        conexao_sqlite
    )

totais_csv = pd.DataFrame({
    "transacoes": [csv_segmento["total_transacoes"].sum()],
    "fraudes": [csv_segmento["qtd_fraudes"].sum()],
    "valor_em_risco": [
        round(csv_segmento["valor_em_risco"].sum(), 2)
    ]
})

reconciliacao_exportacao = pd.DataFrame({
    "Indicador": [
        "Transações",
        "Fraudes",
        "Valor em risco"
    ],
    "CSV": [
        totais_csv.loc[0, "transacoes"],
        totais_csv.loc[0, "fraudes"],
        totais_csv.loc[0, "valor_em_risco"]
    ],
    "SQLite": [
        totais_sqlite.loc[0, "transacoes"],
        totais_sqlite.loc[0, "fraudes"],
        totais_sqlite.loc[0, "valor_em_risco"]
    ]
})

reconciliacao_exportacao["Validação"] = (
    reconciliacao_exportacao["CSV"].round(2)
    == reconciliacao_exportacao["SQLite"].round(2)
).map({
    True: "OK",
    False: "DIVERGENTE"
})

reconciliacao_exportacao

,Indicador,CSV,SQLite,Validação
0,Transações,100000.00,100000.00,OK
1,Fraudes,1833.00,1833.00,OK
2,Valor em risco,314587.93,314587.93,OK


In [10]:
decisao_serving = pd.DataFrame({
    "Alternativa": [
        "CSV",
        "SQLite"
    ],
    "Vantagem": [
        "Portátil, simples e pode ser lido diretamente pelo Plotly",
        "Permite consultas SQL e integração estruturada"
    ],
    "Limitação": [
        "Não oferece consultas concorrentes nem controle transacional",
        "Exige conexão com o arquivo do banco"
    ],
    "Uso definido": [
        "Dashboard do Lab 11",
        "Demonstração de serving relacional"
    ]
})

decisao_serving

,Alternativa,Vantagem,Limitação,Uso definido
0,CSV,"Portátil, simples e pode ser lido diretamente ...",Não oferece consultas concorrentes nem control...,Dashboard do Lab 11
1,SQLite,Permite consultas SQL e integração estruturada,Exige conexão com o arquivo do banco,Demonstração de serving relacional


In [11]:
conexao_duckdb.close()

print("Conexão encerrada.")
print("Lab 9 executado com sucesso.")

Conexão encerrada.
Lab 9 executado com sucesso.


## Conclusão

As duas tabelas Gold foram disponibilizadas fora do ambiente analítico por dois caminhos distintos.

No primeiro, os indicadores foram exportados diretamente para CSV. Essa alternativa é simples, portátil e adequada ao dashboard em Plotly, que será desenvolvido posteriormente.

No segundo, as tabelas foram carregadas em um banco SQLite, simulando a disponibilização da camada Gold em um banco relacional de serving. Esse caminho permite que aplicações externas consultem os dados por SQL.

As contagens e os principais indicadores foram reconciliados entre CSV e SQLite, demonstrando que a exportação não provocou perda ou alteração dos resultados. Para o Lab 11, foi escolhido o CSV por não exigir servidor, conexão administrativa ou configuração adicional.